# SQL Query Practice — Aggregation & Joins

Notes from SQL practice: `COUNT` variants, `WHERE` vs `HAVING`, JOIN logic, and own-goal handling.

**How to revise:** each topic has a notes cell (question + concept) above its query cell (the SQL). The `world`, `game`, `goal`, `team`, and `player` tables are practice tables.


## Schema

**3-table version** (game, team, goal) — used for the JOIN question:

* `game(id, played, city, team1, team2)` — team1/team2 = the two teams playing
* `team(id, teamname, coach)`
* `goal(game, team, player, gtime)`

**4-table version** (adds player) — needed for the own-goal question:

* `player(team, playername, pos)`

**Join keys:**

* `goal.game` → `game.id`
* `goal.team` → `team.id` (team CREDITED with the goal)
* `player.playername` → `goal.player`, and `player.team` = player's ACTUAL team
* `game.team1` / `game.team2` → `team.id` (the two teams in that game)

**world table** (for the COUNT / WHERE vs HAVING questions): `world(name, continent, area, population, gdp)`.

## 1. COUNT(*) vs COUNT(column)

**Question:** Count countries with area >= 1,000,000.

* `COUNT(*)` counts all rows (includes rows where other columns are NULL).
* `COUNT(column)` counts only non-NULL values in that specific column.
* Interview takeaway: `COUNT(*)` = total rows; `COUNT(column)` = non-NULL values in that column.

In [0]:
%sql
-- Counts all rows matching the filter
SELECT COUNT(*) FROM world WHERE area >= 1000000;

-- Counts only rows where name is NOT NULL
SELECT COUNT(name) FROM world WHERE area >= 1000000

## 2. WHERE vs HAVING

**Question:** Count countries with population >= 10 million per continent.

* `WHERE` filters rows before grouping — use for individual row conditions.
* `HAVING` filters groups after aggregation — use for aggregate conditions (SUM, COUNT, AVG).
* Interview distinction: WHERE = row filter; HAVING = group filter.

In [0]:
%sql
-- WHERE: filter rows first, then group
SELECT continent, COUNT(name)
FROM world
WHERE population >= 10000000
GROUP BY continent;

-- HAVING: group first, then filter groups (different question)
SELECT continent, COUNT(name)
FROM world
GROUP BY continent
HAVING SUM(population) >= 100000000

## 3. Multiple JOINs use AND logic

**Question:** List players and teams for goals scored in Vancouver.

* Multiple JOINs apply AND logic — a row must satisfy ALL join conditions.
* Each JOIN adds another condition that must be TRUE.
* For OR logic: use WHERE clause with OR, or UNION separate queries.

In [0]:
%sql
-- All three conditions must be TRUE (AND logic)
SELECT g1.player, t.teamname
FROM game g2
JOIN goal g1 ON g1.game = g2.id      -- AND
JOIN team t ON t.id = g1.team        -- AND
WHERE g2.city = 'Vancouver'          -- AND

## 4. Own-Goals: goal.team vs player.team

**Question:** Understanding which team to show for a goal.

* `goal.team` = team credited with the goal (who got the point).
* `player.team` = player's actual team (who they play for).
* They differ in own-goals: a German player scoring for Brazil means goal.team = Brazil, player.team = Germany.
* Interview takeaway: know which team the question asks for — credited or actual.

In [0]:
%sql
-- Shows team CREDITED (wrong for "player's team" questions)
SELECT g1.player, t.teamname
FROM goal g1
JOIN team t ON t.id = g1.team;

-- Shows player's ACTUAL team (correct for "player's team")
SELECT g1.player, p.team
FROM goal g1
JOIN player p ON p.playername = g1.player

## Key Concepts Summary

| Concept | Rule / Distinction |
| --- | --- |
| COUNT(*) vs COUNT(column) | COUNT(*) = all rows; COUNT(column) = non-NULL values only |
| WHERE vs HAVING | WHERE filters rows before grouping; HAVING filters groups after aggregation |
| Multiple JOINs | Use AND logic by default — all conditions must be TRUE |
| OR logic in JOINs | Use WHERE clause with OR, or UNION separate queries |
| Own-goals | goal.team = credited team; player.team = actual team (differ in own-goals) |